# Test MiDaS Batch Size and Worker Count

Benchmarks `MiDaSAdapter.predict_batch` over a fixed sample of KITTI-C image paths. It reports throughput, latency, and peak CUDA memory for each `(BATCH_SIZE, NUM_WORKERS)` combination.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'configs' / 'dataset_paths.yaml').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root from the current working directory')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)


Project root: /home/kaiyul3/cs543


In [2]:
import gc
import time

import pandas as pd
import torch
from tqdm import tqdm

from src.adapters.midas_adapter import MiDaSAdapter

MODEL_TYPE = 'dpt_hybrid_384'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'kitti_c_manifest.csv'
OUT_CSV = PROJECT_ROOT / 'outputs' / 'metrics' / 'batch_worker_benchmark.csv'

# Keep this moderate while tuning. Full-dataset benchmarking is slow and unnecessary.
MAX_SAMPLES = 2048
RANDOM_STATE = 42

# Candidate settings. Include 0 workers to measure sequential loading.
BATCH_SIZES = (256, 320, 384, 448, 512, 1024, 2048)
NUM_WORKERS_LIST = (8, 12, 16)



# Warmup forwards are excluded from timing.
WARMUP_IMAGES = 8


/opt/conda/lib/python3.13/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:
manifest_df = pd.read_csv(MANIFEST_PATH, dtype={'frame_id': str})
manifest_df = manifest_df[manifest_df['image_path'].notna()].reset_index(drop=True)

if MAX_SAMPLES is not None and MAX_SAMPLES < len(manifest_df):
    bench_df = manifest_df.sample(n=MAX_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    bench_df = manifest_df.copy()

image_paths = bench_df['image_path'].tolist()

print('Manifest path:', MANIFEST_PATH)
print('Manifest rows:', len(manifest_df))
print('Benchmark images:', len(image_paths))
print(bench_df[['corruption_type', 'severity', 'sequence', 'frame_id', 'image_path']].head())


Manifest path: /home/kaiyul3/cs543/data/manifests/kitti_c_manifest.csv
Manifest rows: 63427
Benchmark images: 2048
  corruption_type  severity                    sequence    frame_id  \
0  gaussian_noise         1  2011_09_30_drive_0018_sync  0000001391   
1       iso_noise         4  2011_09_26_drive_0101_sync  0000000658   
2      shot_noise         2  2011_10_03_drive_0027_sync  0000004173   
3   impulse_noise         5  2011_09_26_drive_0036_sync  0000000288   
4     motion_blur         5  2011_09_26_drive_0036_sync  0000000512   

                                          image_path  
0  /home/kaiyul3/cs543/data/raw/kitti_c/kitti_c/g...  
1  /home/kaiyul3/cs543/data/raw/kitti_c/kitti_c/i...  
2  /home/kaiyul3/cs543/data/raw/kitti_c/kitti_c/s...  
3  /home/kaiyul3/cs543/data/raw/kitti_c/kitti_c/i...  
4  /home/kaiyul3/cs543/data/raw/kitti_c/kitti_c/m...  


In [4]:
adapter = MiDaSAdapter(model_type=MODEL_TYPE)
print('Device:', adapter.device)

warmup_paths = image_paths[:min(WARMUP_IMAGES, len(image_paths))]
if warmup_paths:
    _ = adapter.predict_batch(warmup_paths, batch_size=min(len(warmup_paths), max(BATCH_SIZES)), num_workers=0)
    if adapter.device.type == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
print('Warmup complete')


/opt/conda/lib/python3.13/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


Model loaded, number of parameters = 123M
[MiDaSAdapter] Model 'dpt_hybrid_384' loaded on cuda
Device: cuda
Warmup complete


In [5]:
def clear_device_cache():
    gc.collect()
    if adapter.device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def benchmark_setting(batch_size, num_workers, paths):
    clear_device_cache()

    start = time.perf_counter()
    try:
        _ = adapter.predict_batch(paths, batch_size=batch_size, num_workers=num_workers)
        if adapter.device.type == 'cuda':
            torch.cuda.synchronize()
            peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
        else:
            peak_memory_gb = None

        elapsed_s = time.perf_counter() - start
        images_per_second = len(paths) / elapsed_s if elapsed_s > 0 else float('inf')
        batches_per_second = (len(paths) / batch_size) / elapsed_s if elapsed_s > 0 else float('inf')

        return {
            'success': True,
            'batch_size': batch_size,
            'num_workers': num_workers,
            'num_images': len(paths),
            'elapsed_s': elapsed_s,
            'images_per_second': images_per_second,
            'batches_per_second': batches_per_second,
            'ms_per_image': 1000.0 / images_per_second,
            'peak_memory_gb': peak_memory_gb,
            'error': None,
        }

    except RuntimeError as exc:
        if 'out of memory' in str(exc).lower() and adapter.device.type == 'cuda':
            torch.cuda.empty_cache()
        return {
            'success': False,
            'batch_size': batch_size,
            'num_workers': num_workers,
            'num_images': len(paths),
            'elapsed_s': None,
            'images_per_second': 0.0,
            'batches_per_second': 0.0,
            'ms_per_image': None,
            'peak_memory_gb': None,
            'error': str(exc),
        }


In [6]:
results = []
settings = [(bs, nw) for bs in BATCH_SIZES for nw in NUM_WORKERS_LIST]

for batch_size, num_workers in tqdm(settings, desc='benchmark settings'):
    result = benchmark_setting(batch_size, num_workers, image_paths)
    results.append(result)

    if result['success']:
        mem = result['peak_memory_gb']
        mem_text = 'n/a' if mem is None else f'{mem:.2f} GB'
        print(
            f"bs={batch_size:>3} workers={num_workers:>2}: "
            f"{result['images_per_second']:.2f} img/s, "
            f"{result['ms_per_image']:.1f} ms/img, peak={mem_text}"
        )
    else:
        print(f"bs={batch_size:>3} workers={num_workers:>2}: FAILED - {result['error']}")

results_df = pd.DataFrame(results)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUT_CSV, index=False)
print('Saved:', OUT_CSV)
results_df


benchmark settings:   5%|▍         | 1/21 [00:18<06:00, 18.00s/it]

bs=256 workers= 8: 114.60 img/s, 8.7 ms/img, peak=14.59 GB


benchmark settings:  10%|▉         | 2/21 [00:31<04:55, 15.57s/it]

bs=256 workers=12: 149.35 img/s, 6.7 ms/img, peak=14.59 GB


benchmark settings:  14%|█▍        | 3/21 [00:45<04:22, 14.57s/it]

...

benchmark settings:  95%|█████████▌| 20/21 [03:31<00:13, 13.26s/it]

bs=2048 workers=12: FAILED - upsample_bilinear2d_nhwc only supports output tensors with less than INT_MAX elements, but got [2048, 256, 64, 192]


benchmark settings: 100%|██████████| 21/21 [03:47<00:00, 10.82s/it]

bs=2048 workers=16: FAILED - upsample_bilinear2d_nhwc only supports output tensors with less than INT_MAX elements, but got [2048, 256, 64, 192]
Saved: /home/kaiyul3/cs543/outputs/metrics/batch_worker_benchmark.csv


,success,batch_size,num_workers,num_images,elapsed_s,images_per_second,batches_per_second,ms_per_image,peak_memory_gb,error
0,True,256,8,2048,17.870805,114.600323,0.447658,8.725979,14.594881,NaN
1,True,256,12,2048,13.713013,149.347188,0.583387,6.695807,14.594393,NaN
2,True,256,16,2048,13.215189,154.973193,0.605364,6.452729,14.594027,NaN
3,True,320,8,2048,16.436931,124.597470,0.389367,8.025845,18.068880,NaN
4,True,320,12,2048,14.636690,139.922346,0.437257,7.146821,18.067720,NaN
5,True,320,16,2048,37.983885,53.917602,0.168493,18.546819,18.069490,NaN
6,False,384,8,2048,NaN,0.000000,0.000000,NaN,NaN,upsample_bilinear2d_nhwc only supports output ...
7,False,384,12,2048,NaN,0.000000,0.000000,NaN,NaN,upsample_bilinear2d_nhwc only supports output ...
8,False,384,16,2048,NaN,0.000000,0.000000,NaN,NaN,upsample_bilinear2d_nhwc only supports output ...
9,False,448,8,2048,NaN,0.000000,0.000000,NaN,NaN,upsample_bilinear2d_nhwc only supports output ...


In [7]:
successful_results = results_df[results_df['success']].copy()

if successful_results.empty:
    raise RuntimeError('All benchmark settings failed. Check the errors in results_df.')

best_result = successful_results.sort_values('images_per_second', ascending=False).iloc[0]

print('Best setting:')
print('BATCH_SIZE =', int(best_result['batch_size']))
print('NUM_WORKERS =', int(best_result['num_workers']))
print('Images/sec =', round(float(best_result['images_per_second']), 2))
print('ms/image =', round(float(best_result['ms_per_image']), 2))
if pd.notna(best_result['peak_memory_gb']):
    print('Peak GPU memory GB =', round(float(best_result['peak_memory_gb']), 2))

successful_results.sort_values('images_per_second', ascending=False).head(10)


Best setting:
BATCH_SIZE = 256
NUM_WORKERS = 16
Images/sec = 154.97
ms/image = 6.45
Peak GPU memory GB = 14.59


,success,batch_size,num_workers,num_images,elapsed_s,images_per_second,batches_per_second,ms_per_image,peak_memory_gb,error
2,True,256,16,2048,13.215189,154.973193,0.605364,6.452729,14.594027,NaN
1,True,256,12,2048,13.713013,149.347188,0.583387,6.695807,14.594393,NaN
4,True,320,12,2048,14.636690,139.922346,0.437257,7.146821,18.067720,NaN
3,True,320,8,2048,16.436931,124.597470,0.389367,8.025845,18.068880,NaN
0,True,256,8,2048,17.870805,114.600323,0.447658,8.725979,14.594881,NaN
5,True,320,16,2048,37.983885,53.917602,0.168493,18.546819,18.069490,NaN
